# Setup

Install DaliScope following the root README before launching this notebook and select the **Python (DaliScope)** kernel. If packages were updated in this session, restart the kernel once. Run cells in order. The cells below locate the repository and set the working directory to `notebooks` for the relative data paths.

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd().resolve()
if not (ROOT / "daliscope").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "daliscope").is_dir():
    raise RuntimeError("Start in the DaliScope root or notebooks directory")
os.chdir(ROOT / "notebooks")
import daliscope
print("DaliScope", daliscope.__version__)

In [ ]:
import pandas as pd
from IPython.display import display

import daliscope
import daliscope.core.project
import daliscope.optics.histograms
import daliscope.analysis.data_preview
import daliscope.analysis.occupancy
import daliscope.analysis.structal_similarity # for cluster evaluation
import daliscope.widgets.pairwise_py3Dmol
import daliscope.optics.modules_sunburst
import daliscope.analysis.communities
import daliscope.analysis.pfam_stats

from daliscope.analysis.data_preview import show_heading

# Worked Example: Fish-Eye View of Fold Space Around a GOLD-like Domain with Jelly-Roll Topology

*Using DaliScope functions*

## Background

GOLD domains (Pfam clan CL0521) are small β-sandwich modules — eight strands arranged in two sheets, adopting a jelly-roll topology related to the Nucleoplasmin-like/VP fold — found across a functionally coherent family of proteins involved in Golgi trafficking. We use a representative GOLD-like domain as the query for this worked example: PDB entry 5TDQ, chain A, Golgi resident protein GCP60.

## Search strategy

Starting from this query, we searched AFDB2 using ISS-ProtSci with transitive closure, returning over 12,000 hits — far too many, and far too structurally heterogeneous, to inspect one at a time. As a first step, we modeled the *core* fold as a coherent set of secondary structural elements present across more than 5,000 of these hits, giving us a working definition of "the jelly-roll" independent of any single hit's idiosyncrasies.

## The fish-eye problem

A standard Dali search reports only one thing per hit: a Z-score, the structural similarity of that hit *to the query*. This is a radial measurement — it tells you how far each point sits from the center, but nothing about how the points relate to *each other*. The resulting picture is a fish-eye view: everything is placed relative to a single vantage point, and two hits that are highly similar to one another can appear on opposite sides of the query with no indication that they're related at all. This is a real limitation once a fold turns out — as this one does — to be shared across many distinct superfamilies: radial distance from the query cannot, by construction, distinguish "many independent instances of the same superfamily" from "several structurally distinct superfamilies that each merely resemble the query."

## Approach: the Rigid Core Pipeline

To move beyond this single-vantage-point view, we ran DaliScope's Rigid Core Pipeline: pairwise STRUCTAL rescoring among the hits themselves (not just against the query), followed by Infomap community detection on the resulting similarity network. This identifies communities of mutually similar sub-populations directly from the geometry of the hit set, with no reference to functional annotation. We then use Pfam classification, held out from the clustering itself, to ask whether these purely geometric communities are functionally coherent — and show that they are, with individual communities enriched for specific, related Pfam annotations.

## What this document covers

Working from this community structure, we:

- visualize the overall hit landscape and a Pfam-colored scatterplot of the search results;
- filter to the domain-relevant population and run the STRUCTAL/Infomap community analysis described above;
- summarize each community's characteristic Pfam annotations and free-text description stems.

Taken together, the contrast between functional coherence at the module level and the presence or absence of this signature motif within modules lets us formulate concrete functional hypotheses for hundreds of currently unannotated target proteins. Beyond the specific biological result, this worked example is intended primarily to demonstrate DaliScope's analytical and visualization capabilities end to end — from a raw multi-thousand-hit search result to a structured, testable functional hypothesis.

# Raw Dali data
On loading the data, DaliScope presents a static scatterplot in the (Z-score, query-coverage) plane. These are the two main measures of alignment quality. They are generally correlated. The Dali-score is a sum over similarities of intramolecular distances - a more precise fit gives a higher Z-score with the same number of aligned residues.

The plot uses colour coding for 
- sequence identity regimes
- Z-score regimes

Z-score regimes are based on heuristic thresholds - they usually work for "typical cases". Emprically, Dali Z-scores rank hits in reasonable agreement with expert-curated manual classifications like SCOP. However, the optimal classification thresholds differ from one query structure to another, depending on the shape and size of the protein.

Sequence identity regimes indicate what could be found by traditional sequence search (homologs), where structural alignment improves alignment quality (twilight) and where it is indispensable (dark).

In [ ]:
# Load data
# The data pack must be uploaded to the Notebook host. Default data files are in DaliScope/notebooks/data.
pack_path = 'data/GOLD.tar.gz'
if not Path(pack_path).is_file():
    raise FileNotFoundError('GOLD pack is optional. See data/README.md for release download instructions.')

# 1. Initialize & Preview Top Hits
project, FULL_DF, FULL_VIEW = daliscope.analysis.data_preview.load_and_inspect_project(pack_path)

# 2. Plot Landscape
fig = daliscope.analysis.data_preview.plot_dali_hits_with_boundaries(
    FULL_DF, 
    daliscope.analysis.data_preview.build_and_register_subsets(project, FULL_DF, FULL_VIEW)
)

>The plot shows a typical mix of sequence identity regimes: homologs have the highest Z-scores, followed by twilight hits. The large peak with dark hits can contain genuine biologically interesting hits, and noise.
>
>DaliScope automatically created two subsets wihch bisect the plane at valleys in the density distribution. Sometimes - but far from always - the subsets correspond to families or superfamilies. These automatically generated subsets are called FILTERED_0 and FILTERED_1.

# Structural coherence weighted alignment for rigid-body superimposition
Dali alignments optimize the similarity of intramolecular distances, which often results in high RMSD values. This is because far off CA atoms can be at the same radial distance from the common core, but occupy radically different spatial locations which destroy the rigid-body superimposition.We address this problem by weighted superimposition. Firstly, domain clipping gives zero weight to any atom pairs that were aligned by Dali but are outside the domain. Secondly, within the domain, positions are weighted by occupancy so that irregular loops, for exmaple, do not drive the superimposition. 

The stacked alignment coloured by secondary structure (blue helices, red strands, green loops). Multidomain proteins typically display rectangular blocks corresponding to mobile modules. 

We use this plot as a reference for defining domain boundaries: residues 23-123 form a coherent structural core. The "dom" view contains the clipped alignments and recomputed alignment metrics. We also generate a "dom_FOLD" subset, which consists of targets that cover at least 80& of the structural core.

In [ ]:
# 3. Project structural alignments onto Query positions
project.viz.plot_msa(FULL_VIEW, 0, 123, plot_type='heatmap', data_col='dssp_pileup')

# 2. Register clipped parent domain. Updates superimpositions and alignment metrics (except Z-score which stays global)
daliscope.mechanics.metrics.register_multiple_domains(domain_string="23-123", project=project, custom_name="dom")

# 3. Register filtered FOLD subset. 
daliscope.analysis.occupancy.create_fold_subset(project, 'dom', show_plot=False, cutoff=0.8)

# 4. Compare rmsd in raw Dali data, after domain clipping, and after FOLD ocuupancy thresholding
df = FULL_DF[['target_id','rmsd']].copy()
df['clipped & filtered'] = project.views['dom_FOLD']['rmsd']
df['clipped'] = project.views['dom']['rmsd']
daliscope.optics.histograms.plot_overlaid_distributions(
    [df['rmsd'], df['clipped'], df['clipped & filtered']], 
    figsize=(6,2),
    title='RMSD distribution'
)

# Play with the structures
The Query structure is blue, the target is red. Thick lines and dashed lines show the raw Dali alignment, but the least-quares superimposition is based on the clipped and weighted equivalences.

In [ ]:
daliscope.widgets.pairwise_py3Dmol.generate_superimposition_view(
    target_ix=2777,
    df=project.views['dom_FOLD'],
    project=project,
    q_ca_coords=project.query_ca_coords,
    target_ca_coords=project.coords,
)

# What do existing domain classifications tell us?
Here, we show the Pfam family and superfamily (clan) assignments of the domain-fold populations.

The table links family and clan identifiers to InterPro documentation. The clan descriptions confirm that these populations include jellyroll folds, a common structural topology.

A notable fraction of the targets lack Pfam assignments despite structural similarity to the domain fold, with approximately one thousand targets in the “Unassigned” category. Such gaps in domain classification occur even among targets with high Z-scores. In the domain cartoons, these targets appear as empty dashed red boxes; in the scatterplot, they appear as green dots dispersed among known GOLD-like domains (CL051).

These gaps in domain classification motivate a complementary question: **what structural communities exist within the domain-fold population, independent of sequence or existing annotation?**

In [ ]:
# 1. Table view
daliscope.analysis.pfam_stats.pfam_counts('dom_FOLD', project, k=10, print_tables = True)

In [ ]:
# 2. Domain architecture cartoons
project.viz.architecture('dom_FOLD', renderer='matplotlib', pfam_col='pfam_domains')

In [ ]:
# 3. Scatterplot colored by Pfam categories
df = project.views['dom_FOLD'][['alignment_length', 'z_score', 'clan', 'pfam']].copy()

#df["pfam_label"] = daliscope.analysis.pfam_stats.top_category_label(df, "pfam", k=10)
df["clan_label"] = daliscope.analysis.pfam_stats.top_category_label(df, "clan", k=10)

daliscope.optics.histograms.plot_scatter(df, x='z_score', y='alignment_length', color='clan_label', engine='matplot',)

# Community detection

In [ ]:
# Run Rigid Core Pipeline

# Call community detection function across dom_FOLD
modules_df, n_modules = daliscope.analysis.communities.run_community_detection(
    view_name="dom_FOLD",
    domain_range=(23, 123),
    project=project,
    k_neighbors=200,
    max_workers=1,
    verbose=True,
)

# Sunburst diagrams. Colors are recycled after 18 clans
fig, axes, clan_colors = daliscope.optics.modules_sunburst.plot_module_sunbursts(
    modules_df,
    modules=range(1, n_modules+1),
)

# Cross-tabulating pfam catefories and modules
daliscope.analysis.communities.module_pfam_table(project, modules_df, row_group='clan', min_count=20)
daliscope.analysis.communities.module_pfam_table(project, modules_df, row_group='pfam', min_count=20)

## Recursive splitting
To test whether a module represents a single cohesive fold or a composite of finer structural sub-types, we apply community detection recursively to isolated module subsets. This approach reveals two distinct structural behaviors:

- dom_FOLD_module_12 (Monolithic Module): The structural network remains indivisible, confirming a single, tightly conserved structural core. This structural cohesion makes the unassigned targets strong candidates for unification with CL0100 (C1q_TNF).

- dom_FOLD_module_13 (Composite Module): The internal alignment network splits into distinct sub-clusters that cleanly partition along Pfam functional categories. Recursive analysis isolates PF17065 (a putative cytokine) as structurally distinct from CL0004 (Concanavalin), resolving hidden substructure that was masked at the global level.

In [ ]:
show_heading("dom_FOLD_module_12")
daliscope.optics.modules_sunburst.module_recursion(project, 12)

In [ ]:
show_heading("dom_FOLD_module_13")
daliscope.optics.modules_sunburst.module_recursion(project, 13)

## How structurally distinct are the modules?
To evaluate module independence across the whole population, we compare the within- and between-module distributions of STRUCTAL scores. While this population-level metric confirms that within-module alignments are significantly elevated over background, global score comparisons carry the caveat that class imbalance can obscure internal community substructure—an aspect directly resolved by the recursive approach above.

In [ ]:
# Statistical evaluation of pairwise STRUCTAL scores between and within modules
_summary_df, raw_scores = daliscope.analysis.structal_similarity.evaluate_module_cohesion(
    project, 'dom_FOLD', modules_df,
    n_within=2000, n_between=8000,
    modules=None,  # or None for everything, with the cost warning
)

# Box-plot
fig, ax = daliscope.analysis.communities.plot_module_score_distributions(
    raw_scores=raw_scores,
    figsize=(12, 6),
    title="Within- and between-module STRUCTAL scores (2000 + 8000 randomly sampled)",
)

# What structural and functional features characterize a population?
We demonstrate structural superimposition, domain architecture cartoons, and summaries of short functional descriptions.

## Transitive superimposition
Here we visualize examples from module 16, which is composed predominantly of germins (PF0100). The germins are structurally similar to one another, while also containing auxiliary structural elements and deletions relative to the Query.

The Query is a GOLD-like domain and is shown as a thin gray line, providing a reference for the structural comparisons.

Mouse controls: rotate, zoom, and pan.

In [ ]:
# Select view. Select targets by index.
view_name = 'dom_FOLD_module_16'
target_ixes = [11, 21, 31, 41, 51]

# Show multiple traces in py3Dmol
view = daliscope.widgets.pairwise_py3Dmol.generate_transitive_superimposition_view(
    target_ixes=target_ixes,
    df=project.views[view_name],
    project=project,
    target_ca_coords=project.coords,
    q_ca_coords=project.query_ca_coords,
    query_radius=0.15,
    show_query=True,
)
view.show()

## Summarizing functional descriptions per module (t.b.i.)
For each module, we select representative functional descriptions using a TF-IDF-based approach.

This section is a planned extension and is not executed in this notebook.

# Summary of Workflow

In this worked example, we demonstrated how **DaliScope** moves beyond basic homology searching:

1. **Beyond Top Hits:** We explored deep structural alignments rather than relying solely on the single highest-scoring match.
2. **User-Driven Decisions:** You retained full control throughout the workflow—from clipping domains and defining active/binding sites to cross-referencing live InterPro annotations.
3. **Reproducible Output:** Because this run relies on a self-contained, frozen data pack, re-running these notebook cells will produce reproducible analysis with the recorded software environment and parameters.